# Module `validators.py`

Ce notebook detaille toutes les validations appliquees au projet. Elles sont separees du generateur pour que chaque brique reste responsable d'une seule chose.

Le module expose :

- des helpers bas niveau (`density_from_edge_count`, `average_degree_from_edge_count`, `is_connected`) ;
- `InstanceValidator` : juge de la validite d'une **instance statique** apres generation ;
- `DynamicStateValidator` : juge de la validite d'un **snapshot dynamique** avant d'autoriser une modification.

Important : ces validateurs ne verifient **pas** la faisabilite d'une solution VRP. Ils verifient la qualite du graphe fourni aux solveurs.

In [ ]:
import sys
from pathlib import Path
from pprint import pprint

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.dynamic_network import DynamicNetworkSimulator
from cesipath.graph_generator import GraphGenerator
from cesipath.models import EdgeStatus, GraphGenerationConfig
from cesipath.validators import (
    DynamicStateValidator,
    InstanceValidator,
    average_degree_from_edge_count,
    density_from_edge_count,
    is_connected,
)

## 1. Helpers bas niveau

Trois fonctions pures servent de briques a tout le reste :

- `density_from_edge_count(n, m) = m / (n*(n-1)/2)` : densite d'un graphe non oriente.
- `average_degree_from_edge_count(n, m) = 2m / n` : degre moyen non oriente (chaque arete compte dans 2 sommets).
- `is_connected(n, edges)` : BFS depuis le sommet 0 ; renvoie `True` si tous les sommets sont atteints.

Les deux premieres sont des formules directes mais elles sont exposees pour etre reutilisables en debug.

In [ ]:
n = 6
edges = {(0, 1), (1, 2), (2, 3), (3, 4), (4, 5)}  # chaine
print("Densite   :", round(density_from_edge_count(n, len(edges)), 3))
print("Degre moy :", round(average_degree_from_edge_count(n, len(edges)), 3))
print("Connexe ? :", is_connected(n, edges))

edges_broken = {(0, 1), (2, 3)}
print("Connexe (casse) ?", is_connected(n, edges_broken))

## 2. `InstanceValidator` - 4 criteres statiques

Appele par `GraphGenerator.generate()` dans la boucle de rejet. Une instance est valide **ssi tous les criteres ci-dessous passent** :

| # | Critere | Source |
|---|---|---|
| 1 | `base_density` dans `[resolved_min_base_density, resolved_max_base_density]` | toutes aretes de `base_edges` |
| 2 | `residual_density` dans `[resolved_min_residual_density, resolved_max_residual_density]` | aretes non FORBIDDEN de `residual_edges` |
| 3 | `residual_average_degree >= resolved_min_average_residual_degree` | aretes non FORBIDDEN |
| 4 | Graphe residuel connexe apres retrait des FORBIDDEN | `is_residual_graph_connected` |

Si un seul critere echoue, le generateur rejette l'instance et recommence avec une nouvelle graine interne.

In [ ]:
config = GraphGenerationConfig(node_count=8, seed=42)
instance = GraphGenerator(config).generate()
validator = InstanceValidator(config)

print("Instance valide ?", validator.is_valid(instance))
print()
print("  base_density        :", round(instance.base_density, 4),
      "in [", config.resolved_min_base_density, ",", config.resolved_max_base_density, "]")
print("  residual_density    :", round(instance.residual_density, 4),
      "in [", config.resolved_min_residual_density, ",", config.resolved_max_residual_density, "]")
print("  residual_avg_degree :", round(instance.residual_average_degree, 4),
      ">=", config.resolved_min_average_residual_degree)
print("  residuel connexe    :", InstanceValidator.is_residual_graph_connected(instance))

## 3. `DynamicStateValidator` - 4 invariants dynamiques

Appele par `DynamicNetworkSimulator.advance()` **avant** chaque coupure d'arete. Si l'un des invariants casserait, la coupure est refusee et l'arete reste active.

| # | Invariant | Source |
|---|---|---|
| 1 | Graphe actif connexe | `is_connected` sur `edge_availability` |
| 2 | `active_density >= resolved_dynamic_min_density` | densite des aretes ON |
| 3 | `active_average_degree >= resolved_dynamic_min_average_degree` | degre moyen des aretes ON |
| 4 | `disabled_ratio <= dynamic_max_disabled_ratio` | part d'aretes OFF par rapport au total dynamique |

Ces invariants garantissent qu'un solveur recevra toujours un graphe complet et suffisamment dense, meme au pire moment de la simulation.

In [ ]:
simulator = DynamicNetworkSimulator(instance, seed=42)
snapshot = simulator.initialize_snapshot()
dynamic_validator = DynamicStateValidator(instance)

candidate = next(iter(snapshot.edge_availability))
print("Arete candidate :", candidate)
print("Peut-on la passer OFF ?", dynamic_validator.can_disable_edge(snapshot.edge_availability, candidate))

## 4. Contre-exemple : casser la connexite

Pour illustrer la robustesse, on cree manuellement un graphe en chaine `0-1-2-3`, on met toutes les aretes sur `ON`, puis on tente de couper l'arete centrale `(1, 2)`. Cela couperait le graphe en deux, le validateur doit refuser.

In [ ]:
# On construit un petit instance artificielle via l'instance reelle puis on forge les aretes.
from dataclasses import replace

config_chain = GraphGenerationConfig(node_count=4, seed=1, auto_density_profile=False,
                                     min_base_density=0.0, max_base_density=1.0,
                                     min_residual_density=0.0, max_residual_density=1.0,
                                     min_average_residual_degree=0.0,
                                     dynamic_max_disabled_ratio=0.5)
instance_chain = GraphGenerator(config_chain).generate()

# On force une configuration en chaine : seules (0,1), (1,2), (2,3) sont actives.
chain = {(0, 1), (1, 2), (2, 3)}
availability = {key: (key in chain) for key in instance_chain.residual_edges}

dv = DynamicStateValidator(instance_chain)
print("On veut couper (1, 2) :",
      dv.can_disable_edge(availability, (1, 2)))
print("On veut couper (0, 1) :",
      dv.can_disable_edge(availability, (0, 1)))

## 5. Resume mental

- `InstanceValidator` protege la **qualite** du graphe statique (densite raisonnable, connexe).
- `DynamicStateValidator` protege la **stabilite** de la simulation (pas de partition, pas de vidage progressif du reseau).
- Les deux lisent les seuils `resolved_*` de la config : on change un seuil, les validateurs suivent automatiquement.